In [0]:
bearscash_invoice_staging=dbutils.widgets.get("bearscash_invoice_staging")
bearscash_invoice=dbutils.widgets.get("bearscash_invoice")
cash_invoice=dbutils.widgets.get("cash_invoice")
cash_stg=dbutils.widgets.get("cash_stg")
bears_cashhistory=dbutils.widgets.get("bears_cashhistory")
cubeserviceofficetxnweekendingdate=dbutils.widgets.get("cubeserviceofficetxnweekendingdate")
payerdimension=dbutils.widgets.get("payerdimension")
office=dbutils.widgets.get("office")
cubeserviceofficetxnsourcesystem=dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
client=dbutils.widgets.get("client")
cubeserviceofficetxnpayorcategory=dbutils.widgets.get("cubeserviceofficetxnpayorcategory")
datedimension=dbutils.widgets.get("datedimension")
mart_cash=dbutils.widgets.get("mart_cash")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {cash_invoice}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {cash_invoice} executed")
    spark.sql(f"""
    INSERT INTO {cash_invoice}
    (
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    client_key,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    agency_id,
    payor_category,
    invoice_number,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    deposit_number,
    hchb_date_entered_key,
    month_end_close_reporting_period
    )
    SELECT
    CAST(Source_System_Key AS INT),
    CAST(Reporting_Week_Ending_Date_Key AS INT),
    CAST(Posted_Date_Key AS INT),
    CAST(Payor_Key AS INT),
    CAST(Client_Key AS INT),
    CAST(Cash_Collected AS DECIMAL(15,2)),
    CAST(Deposit_Date_Key AS INT),
    CAST(Batch_Id AS STRING),
    CAST(Check_Id AS STRING),
    CAST(Office_Key AS INT),
    CAST(Type AS STRING),
    CAST(Bank AS STRING),
    CAST(Agency_Id AS INT),
    CAST(Payor_Category AS STRING),
    CAST(Invoice__ AS STRING),
    CAST(Rap_Payment AS DECIMAL(19,4)),
    CAST(Final_Payment AS DECIMAL(19,4)),
    CAST(Other_Payment AS DECIMAL(19,4)),
    CAST(Refund_Payment AS DECIMAL(19,4)),
    CAST(Unapplied_Cash AS DECIMAL(19,4)),
    CAST(Deposit__ AS STRING),
    CAST(HCHB_Date_Entered_Key AS INT),
    CAST(Month_End_Close_Reporting_Period AS STRING)
    FROM {mart_cash}
    WHERE Source_System_Key IN (0, 19);

    """)

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW Bearscash AS
SELECT 
  CASE 
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 1 THEN TO_DATE(PostedDate, 'MM-dd-yy')
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 2 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -1)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 3 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -2)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 4 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), -3)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 5 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 3)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 6 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 2)
    WHEN DAYOFWEEK(TO_DATE(PostedDate, 'MM-dd-yy')) = 7 THEN DATE_ADD(TO_DATE(PostedDate, 'MM-dd-yy'), 1)
  END AS ReportingWeekendingDate,
  'BEARS' AS SourceSystem,
  PayorTypeCode,
  OfficeNumber,
  SUM(AppliedOnMR) * -1.00 AS CashCollected,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank
FROM {cash_stg} C
GROUP BY 
  CASE DAYOFWEEK(PostedDate)
    WHEN 1 THEN PostedDate
    WHEN 2 THEN DATE_ADD(PostedDate, -1)
    WHEN 3 THEN DATE_ADD(PostedDate, -2)
    WHEN 4 THEN DATE_ADD(PostedDate, -3)
    WHEN 5 THEN DATE_ADD(PostedDate, 3)
    WHEN 6 THEN DATE_ADD(PostedDate, 2)
    WHEN 7 THEN DATE_ADD(PostedDate, 1)
  END,
  PayorTypeCode,
  OfficeNumber,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank;
""" 
)

spark.sql(
    f"""
INSERT INTO {bears_cashhistory}
SELECT
    reportingweekendingdate,
    sourcesystem,
    payortypecode,
    officenumber AS serviceofficenumber,
    cashcollected,
    payorid,
    billtoname,
    posteddate,
    depositdate,
    batchid,
    checkid,
    type,
    batchnumber,
    bank,
    invoicenumber,
    clientnumber,
    NULL AS product
FROM bearscash
"""
)


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW src_bearscash AS
SELECT 
    SS.SourceSystemKey AS source_system_key,
    W.WeekEndingDateKey AS reporting_week_ending_date_key,
    PD.DateKey AS posted_date_key,
    COALESCE(P.PayerKey, -1) AS payor_key,
    PT.PayorCategoryDescription AS payor_category_description,
    CL.ClientKey AS client_key,
    ca.InvoiceNumber AS invoice_number,
    CAST(ca.CashCollected AS DECIMAL(15,2)) AS cash_collected,
    DD.DateKey AS deposit_date_key,
    ca.BatchID AS batch_id,
    ca.CheckID AS check_id,
    Oc.OfficeKey AS office_key,
    ca.Type AS type,
    ca.Bank AS bank

FROM (
    SELECT 
        CASE 
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 1 THEN cast(PostedDate as date)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 2 THEN date_add(cast(PostedDate as date), -1)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 3 THEN date_add(cast(PostedDate as date), -2)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 4 THEN date_add(cast(PostedDate as date), -3)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 5 THEN date_add(cast(PostedDate as date), 3)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 6 THEN date_add(cast(PostedDate as date), 2)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 7 THEN date_add(cast(PostedDate as date), 1)
        END AS ReportingWeekendingDate,

        'BEARS' AS SourceSystem,
        PayorTypeCode,
        OfficeNumber,
        SUM(AppliedOnMR) * -1.00 AS CashCollected,
        PayorID,
        BillToName,
        ClientNumber,
        cast(PostedDate as date) AS PostedDate,
        cast(DepositDate as date) AS DepositDate,
        BatchID,
        CheckID,
        InvoiceNumber,
        Type,
        BatchNumber,
        Bank

    FROM {cash_stg}
    GROUP BY
        CASE 
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 1 THEN cast(PostedDate as date)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 2 THEN date_add(cast(PostedDate as date), -1)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 3 THEN date_add(cast(PostedDate as date), -2)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 4 THEN date_add(cast(PostedDate as date), -3)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 5 THEN date_add(cast(PostedDate as date), 3)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 6 THEN date_add(cast(PostedDate as date), 2)
            WHEN DAYOFWEEK(cast(PostedDate as date)) = 7 THEN date_add(cast(PostedDate as date), 1)
        END,
        PayorTypeCode,
        OfficeNumber,
        PayorID,
        BillToName,
        ClientNumber,
        cast(PostedDate as date),
        cast(DepositDate as date),
        BatchID,
        CheckID,
        InvoiceNumber,
        Type,
        BatchNumber,
        Bank
) ca

LEFT JOIN {datedimension} PD
    ON PD.CalendarDate = ca.PostedDate

LEFT JOIN {datedimension} DD
    ON DD.CalendarDate = ca.DepositDate

LEFT JOIN {cubeserviceofficetxnweekendingdate} W
    ON W.WeekEndingDate = ca.ReportingWeekendingDate

LEFT JOIN {payerdimension} P
    ON P.PayerID = ca.PayorID

LEFT JOIN (
    SELECT PayorCategoryCode, PayorCategoryKey, PayorCategoryDescription
    FROM {cubeserviceofficetxnpayorcategory}
    WHERE PayorCategoryKey != 46
) PT
    ON ca.PayorTypeCode = PT.PayorCategoryCode

LEFT JOIN {office} Oc
    ON Oc.OfficeNumber = ca.OfficeNumber

LEFT JOIN {cubeserviceofficetxnsourcesystem} SS
    ON SS.SourceSystemName = ca.SourceSystem

LEFT JOIN {client} CL
    ON CL.SourceSystemId = ca.ClientNumber
    AND CL.OfficeNumber = ca.OfficeNumber
    AND CL.SourceSystem = 'BEARS'
""")


In [0]:
spark.sql(
    f"""
INSERT INTO {bearscash_invoice} 
(
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    payor_category_description,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    client_key,
    invoice_number
)
SELECT
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    payor_category_description,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    client_key,
    invoice_number
FROM src_bearscash;

    """
)

In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {bearscash_invoice_staging};
 """
)

spark.sql(
    f"""
INSERT INTO {bearscash_invoice_staging} 
(
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    payor_category_description,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    client_key,
    invoice_number
)
SELECT
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    payor_category_description,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    client_key,
    invoice_number
FROM src_bearscash;

    """
)

In [0]:
spark.sql(
    f"""
INSERT INTO {cash_invoice} (
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    client_key,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    agency_id,
    payor_category,
    invoice_number,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    deposit_number,
    hchb_date_entered_key,
    month_end_close_reporting_period
)
SELECT
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    client_key,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    NULL        AS agency_id,
    payor_category_description AS payor_category,
    invoice_number,
    NULL         AS rap_payment,
    NULL         AS final_payment,
    NULL         AS other_payment,
    NULL         AS refund_payment,
    NULL         AS unapplied_cash,
    NULL        AS deposit_number,
    NULL        AS hchb_date_entered_key,
    NULL        AS month_end_close_reporting_period
FROM {bearscash_invoice_staging}
"""
)
